<a href="https://colab.research.google.com/github/Ryuta-Y/pytorch-practice/blob/main/pytorch_practice_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
%%shell

pip install cython
pip install -U 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'

  Cloning https://github.com/cocodataset/cocoapi.git to /tmp/pip-req-build-fg1_z4qg
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/cocoapi.git /tmp/pip-req-build-fg1_z4qg
  Resolved https://github.com/cocodataset/cocoapi.git to commit 8c9bcc3cf640524c4c20a9c40e89cb6a2f2fa0e9
  Preparing metadata (setup.py) ... done


In [19]:
%%shell

# Penn-Fudanデータセットをダウンロード
wget https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip .
# カレント・フォルダにzipファイルを解凍
unzip PennFudanPed.zip

--2026-04-23 04:39:07--  https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip
Resolving www.cis.upenn.edu (www.cis.upenn.edu)... 158.130.69.163, 2607:f470:8:64:5ea5::d
Connecting to www.cis.upenn.edu (www.cis.upenn.edu)|158.130.69.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53723336 (51M) [application/zip]
Saving to: ‘PennFudanPed.zip.1’

PennFudanPed.zip.1  100%[===================>]  51.23M   134MB/s    in 0.4s    

2026-04-23 04:39:08 (134 MB/s) - ‘PennFudanPed.zip.1’ saved [53723336/53723336]

--2026-04-23 04:39:08--  http://./
Resolving . (.)... failed: No address associated with hostname.
wget: unable to resolve host address ‘.’
FINISHED --2026-04-23 04:39:08--
Total wall clock time: 0.5s
Downloaded: 1 files, 51M in 0.4s (134 MB/s)
Archive:  PennFudanPed.zip
replace PennFudanPed/added-object-list.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

CalledProcessError: Command '
# Penn-Fudanデータセットをダウンロード
wget https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip .
# カレント・フォルダにzipファイルを解凍
unzip PennFudanPed.zip
' returned non-zero exit status 80.

In [ ]:
from PIL import Image
Image.open('PennFudanPed/PNGImages/FudanPed00001.png')

In [ ]:
mask= Image.open('PennFudanPed/PedMasks/FudanPed00001_mask.png')

mask.putpalette([
    0,0,0,
    255,0,0,
    0,255,0,
    0,0,255,
])
mask

In [ ]:
import os
import numpy as np
import torch
import torch.utils.data
from PIL import Image

class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
      self.root=root
      self.transforms=transforms

      self.imgs=list(sorted(os.listdir(os.path.join(root, 'PNGImages'))))
      self.masks=list(sorted(os.listdir(os.path.join(root, 'PedMasks'))))

    def __getitem__(self, idx):
      img_path=os.path.join(self.root, 'PNGImages', self.imgs[idx])
      mask_path=os.path.join(self.root, 'PedMasks', self.masks[idx])
      img=Image.open(img_path).convert('RGB')

      mask=Image.open(mask_path)

      mask=np.array(mask)
      obj_ids=np.unique(mask)
      obj_ids=obj_ids[1:]

      masks=mask==obj_ids[:, None, None]

      num_objs=len(obj_ids)

      boxes=[]
      for i in range(num_objs):
        pos=np.where(masks[i])
        xmin=np.min(pos[1])
        xmax=np.max(pos[1])
        ymin=np.min(pos[0])
        ymax=np.max(pos[0])
        boxes.append([xmin, ymin, xmax, ymax])

      boxes=torch.as_tensor(boxes, dtype=torch.float32)

      labels=torch.ones((num_objs,), dtype=torch.int64)
      masks=torch.as_tensor(masks, dtype=torch.uint8)

      image_id=torch.tensor([idx])
      area=(boxes[:, 3]-boxes[:, 1]) * (boxes[:, 2]-boxes[:, 0])
      iscrowd=torch.zeros((num_objs,), dtype=torch.int64)

      target={}
      target['boxes']=boxes
      target['labels']=labels
      target['masks']=masks
      target['image_id']=image_id
      target['area']=area
      target['iscrowd']=iscrowd

      if self.transforms is not None:
        img, target=self.transforms(img, target)

      return img, target

    def __len__(self):
      return len(self.imgs)

In [ ]:
dataset=PennFudanDataset('PennFudanPed/')
dataset[0]

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model=torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

num_classes=2

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator

# 分類のために訓練済みモデルをロードし、特徴量のみを取得します。
backbone = torchvision.models.mobilenet_v2(pretrained=True).features
# FasterRCNNでは、バックボーンで指定された出力チャネル数を知る必要があります。
# mobilenet_v2の場合は1280なので、ここで1280を設定します。
backbone.out_channels = 1280

# RPN：Resion Proposal Networkに、空間ごとに5 x 3パターンのアンカーを生成させてみましょう。
# これは、アンカーに5つのサイズ（size）と、3つのアスペクト比(aspect_ratio)があることを意味します。
# 特徴マップごとに異なるサイズとアスペクト比となる可能性があるので，Tuple[Tuple[int]] という形式で指定します。
anchor_generator = AnchorGenerator(sizes=((32, 64, 128, 256, 512),),
                                   aspect_ratios=((0.5, 1.0, 2.0),))

# 関心領域のトリミングを実行するために使用する特徴マップ(featmap_names)と、
# 画像の大きさを元に戻した後のトリミングのサイズ(output_size)を定義しましょう。
# バックボーンがTensorを返す場合、featmap_namesは[0]になっているはずです。
# もう少し一般化して説明すると、バックボーンはOrderedDict[Tensor]を返すことになるので、
# featmap_namesで使用する特徴マップを選択できます。
roi_pooler = torchvision.ops.MultiScaleRoIAlign(featmap_names=[0],
                                                output_size=7,
                                                sampling_ratio=2)

# FasterRCNNモデルに上記に定義したパーツをまとめます。
model = FasterRCNN(backbone,
                   num_classes=2,
                   rpn_anchor_generator=anchor_generator,
                   box_roi_pool=roi_pooler)

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def get_instance_segmentation_model(num_classes):
    # COCOデータセットで事前学習したインスタンス・セグメンテーションのモデルをロードします
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')

    # 分類器に入力する特徴量の数を取得します
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # 事前訓練済みのヘッドを新しいヘッドに置き換えます
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # セグメンテーション・マスクの分類器に入力する特徴量の数を取得します
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # セグメテーション・マスクの推論器を新しいものに置き換えます
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask,
                                                       hidden_layer,
                                                       num_classes)

    return model


In [ ]:
%%shell

# 上記ファイルを利用するために、TorchVision のリポジトリをreferences/detectionからダウンロードします
git clone https://github.com/pytorch/vision.git
cd vision
git checkout v0.3.0

cp references/detection/utils.py ../
cp references/detection/transforms.py ../
cp references/detection/coco_eval.py ../
cp references/detection/engine.py ../
cp references/detection/coco_utils.py ../

In [ ]:
import torch
import sys
import types

# torch._sixはPyTorchの新しいバージョンで削除されたため、ダミーのモジュールを作成してエラーを回避します
if not hasattr(torch, '_six'):
    torch._six = types.ModuleType('torch._six')
    torch._six.string_classes = (str,)
    torch._six.int_classes = (int,)
    sys.modules['torch._six'] = torch._six

from engine import train_one_epoch, evaluate
import utils
import transforms as T # transform を transforms に修正

def get_transform(train):
  transforms = []

  transforms.append(T.ToTensor())
  if train:
    transforms.append(T.RandomHorizontalFlip(0.5)) # transfroms を transforms に修正
  return T.Compose(transforms)


In [ ]:
# 作成したカスタム・データセット
dataset = PennFudanDataset('PennFudanPed', get_transform(train=True))
dataset_test = PennFudanDataset('PennFudanPed', get_transform(train=False))

# データセットを訓練セットとテストセットに分割
torch.manual_seed(1)
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-50])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-50:])

# 訓練データと評価データのデータロード用オブジェクトを用意
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, num_workers=4,
    collate_fn=utils.collate_fn)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test, batch_size=1, shuffle=False, num_workers=4,
    collate_fn=utils.collate_fn)

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

num_classes=2

model=get_instance_segmentation_model(num_classes)
model.to(device)

params=[p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler=torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


In [ ]:
num_epochs=10 for epoch in range(num_epochs):
train_one_epoch(model, optimizer, data_loader,device, epoch, print_freq=10)
lr_scheduler.step()
evaluate(model, data_loader_test, device)


In [ ]:
img, _ = dataset_test[0]

model.eval()
with torch.no_grad():
  predicition=model([img.to(device)])

In [ ]:
Image.fromarray(img.mul(255).permute(1,2,0).byte().numpy())
Image.fromarray(prediction[0]['masks'][0,0].mul(255).byte().cpu().numpy())


In [ ]:
print(len(prediction[0]['masks']))
print(prediction[0]['scores'])

In [ ]:
Image.fromarray(prediction[0]['masks'][1,0].mul(255).byte().cpu().numpy())
